# 01: SQL y DuckDB: tu laboratorio de datos 

## Tu primer SELECT: SQL como calculadora


SQL no necesita tablas para funcionar. 

Puedes usar SELECT como una calculadora, para manipular texto o para obtener información del sistema. 

Esto es útil para probar expresiones antes de aplicarlas a datos reales:

```sql
-- SQL como calculadora
SELECT 1 + 1 AS resultado;

-- Manipular texto
SELECT 'Hola' || ' ' || 'mundo' AS saludo;

-- Fecha actual
SELECT CURRENT_DATE AS hoy;

-- Expresiones complejas
SELECT 
    42 * 3.14 AS pi_por_42,
    ROUND(100.0 / 3, 2) AS un_tercio,
    LENGTH('BigDataStack') AS longitud;
```

- SELECT 1 + 1 AS resultado → aritmética básica. AS da un nombre legible a la columna resultante.
- SELECT 'Hola' || ' ' || 'mundo' → el operador || concatena textos en SQL estándar (equivalente al + de strings en Python).
- CURRENT_DATE → función del sistema que devuelve la fecha actual sin hora.
- 42 * 3.14 → aritmética con decimales.
- ROUND(100.0 / 3, 2) → divide y redondea a 2 decimales. Nota: 100.0 (con decimal) fuerza división no-entera.
- LENGTH('BigDataStack') → cuenta los caracteres del texto.

Ninguno usa FROM porque no estamos leyendo datos de una tabla — solo calculando expresiones.

## Explorar las tablas disponibles

Cuando llegas a una base de datos nueva — ya sea en un trabajo nuevo o en esta plataforma — lo primero que haces es explorar: 

1. ¿qué tablas hay? 
2. ¿qué columnas tienen? 
3. ¿cuántos datos hay? 

DuckDB tiene comandos para eso:

```sql
-- Ver todas las tablas disponibles
SHOW TABLES;

-- Ver la estructura de una tabla (columnas y tipos)
DESCRIBE clientes;

-- También funciona con esta sintaxis
DESCRIBE pedidos;
```

- SHOW TABLES → comando de metadata que lista todas las tablas de la base de datos. Es tu primer paso al llegar a un entorno nuevo.
- DESCRIBE clientes → muestra columnas, tipos de datos y restricciones de la tabla. Es como leer el "plano" antes de construir.
- DESCRIBE pedidos → lo mismo para otra tabla.

Estos comandos no modifican datos — solo muestran información sobre la estructura. Son seguros de ejecutar en cualquier momento.

## Tu primer SELECT * FROM


Vamos a consultar datos reales. 

- SELECT * FROM tabla LIMIT n te da las primeras n filas de una tabla. 

Es tu forma de "echar un vistazo" a los datos:



```sql
-- Las primeras 5 filas de la tabla clientes
SELECT * FROM clientes LIMIT 5;

-- Las primeras 5 filas de pedidos
SELECT * FROM pedidos LIMIT 5;

-- Solo columnas específicas
SELECT nombre, ciudad, es_vip,
FROM CLIENTES
LIMIT 10;

-- Cuántas filas tiene una tabla?
SELECT COUNT(*) AS total_filas FROM pedidos;
```

- SELECT * FROM clientes LIMIT 5 → trae TODAS las columnas (*) pero solo 5 filas. Ideal para ver la "forma" de los datos sin descargar miles de filas.
- SELECT * FROM pedidos LIMIT 5 → lo mismo para la tabla de pedidos. Compara estructuras rápidamente.
- SELECT nombre, ciudad, es_vip FROM clientes LIMIT 10 → pide solo 3 columnas. Más eficiente y enfocado que *.
- COUNT(*) → cuenta TODAS las filas de la tabla, sin importar si tienen NULLs. Es tu forma de medir el volumen de datos antes de lanzarte a explorar.

LIMIT es tu red de seguridad al explorar: sin él, una tabla con millones de filas inundaría tu pantalla (y tu memoria).

**Consejo de senior:**

- SELECT * es para explorar. 
- En código que va a producción, siempre lista las columnas explícitamente. 
- He visto pipelines romperse porque alguien añadió una columna de 10MB y el SELECT * empezó a devolver gigas de datos inesperados.


## Declarativo vs imperativo: la clave de SQL


La diferencia fundamental entre SQL y lenguajes como Python es que SQL es declarativo: Dices QUÉ quieres, no CÓMO obtenerlo.

En Python, para encontrar clientes de Madrid escribirías un bucle que recorre cada fila y comprueba la condición. 

En SQL, simplemente describes el resultado:


```sql
-- IMPERATIVO (pseudocódigo Python):
-- resultado = []
-- for fila in clientes:
--      if fila.ciudad == 'Madrid':
--          resultado.append(fila.nombre)

-- DECLARATIVO (SQL):
SELECT nombre
FROM clientes
WHERE ciudad = 'Madrid';
```

Comparación directa entre paradigmas:

- Imperativo (Python): defines cada paso — crear lista vacía, iterar, comprobar condición, añadir al resultado. Tú controlas el CÓMO.
- Declarativo (SQL): SELECT nombre → qué columnas. FROM clientes → de qué tabla. WHERE ciudad = 'Madrid' → qué condición.

El motor SQL decide internamente cómo buscar (¿índice? ¿escaneo completo? ¿paralelismo?). Tú solo dices QUÉ quieres. Es más conciso y el optimizador suele ser más eficiente que un bucle manual.

## Crear tablas y meter datos: DDL y DML


Hasta ahora has consultado tablas que ya existían. 

En tu trabajo vas a necesitar crear tablas propias para: 

1. Pruebas,  
2. Transformaciones intermedias,  
3. Cargar datos nuevos. 

SQL separa dos lenguajes:

1. **DDL (Data Definition Language):** 
    - Para definir estructuras 
2. **DML (Data Manipulation Language):** 
    - Para meter, modificar y borrar datos dentro de esas estructuras.


```sql
-- DDL: crear una tabla con 3 columnas
CREATE TABLE paises (
    id INTERGER PRIMARY KEY, -- identificador
    nombre VARCHAR,          -- texto de longitud variable
    continente VARCHAR
);

-- DML: insertar varias filas de una vez
INSERT INTO paises (id, nombre, continente) VALUES
    (1, 'España', 'Europa'), 
    (2, 'Portugal', 'Europa'),
    (3, 'México', 'América');

-- Comprobar que los datos están ahí
SELECT * FROM paises;
```

**Error típico que verás al principio:** Poner comillas dobles en un texto ("Madrid" en vez de 'Madrid'). 


- SQL interpreta la dobles como un nombre de columna, y el mensaje de error empieza por «Referenced column "Madrid"». 
- En cuanto veas tu texto entre comillas dentro de un mensaje que habla de columnas, ya sabes: cambia las dobles por simples. 
- Aprovecha además la línea de debajo: cuando el error ocurre dentro de un WHERE, DuckDB añade «Candidate bindings:» y te lista las columnas que SÍ existen en la tabla. 
    - Esa línea te va a sacar de casi todos los errores de nombre que cometas de aquí en adelante, no solo de éste.

